# Neural Market Environment: Self-Improving Trading Agent

**Meta PyTorch OpenEnv Hackathon | Theme 3 (World Modeling) + Theme 4 (Self-Improvement)**

We built a neural world model that simulates realistic market dynamics, then trained an LLM agent to trade against it using GRPO (reinforcement learning). The environment generates novel scenarios the agent has never seen, forcing it to learn genuine trading intuition rather than memorize historical data.

**Results:** Base model 0.300 → SFT 0.417 → GRPO **0.537** (on neural environment)

| Component | What it does |
|-----------|-------------|
| Neural World Model | 1.22M param causal transformer, generates OHLCV with 0.94x real volatility |
| LLM Agent | DeepSeek-R1 7B with QLoRA, trades via BUY/SELL/HOLD actions |
| Self-Improvement | SFT (distilled reasoning) → RAFT (rejection sampling) → GRPO (RL against neural env) |
| OpenEnv API | REST + WebSocket, same interface for static and neural environments |

**Stack:** OpenEnv + TRL (GRPOTrainer) + Unsloth + PyTorch

## 1. Setup

Install dependencies and download data from HuggingFace Hub.

In [ ]:
%%capture
# Install dependencies
!pip install -q openenv-core gymnasium torch pandas numpy matplotlib
!pip install -q huggingface_hub datasets

# Clone repo
import os
if not os.path.exists("stock-trader-env"):
    !git clone -q https://github.com/sarthakbiswas97/stock-trader-env.git
os.chdir("stock-trader-env")
!git checkout -q v3/world-model

# Download market data + world model checkpoint
from huggingface_hub import hf_hub_download
from pathlib import Path
from datasets import load_dataset

# Market data
ohlcv_dir = Path("data/ohlcv")
if not ohlcv_dir.exists() or not any(ohlcv_dir.glob("*.csv")):
    ds = load_dataset("sarthakbiswas/stock-trader-market-data")
    ohlcv_dir.mkdir(parents=True, exist_ok=True)
    macro_dir = Path("data/macro")
    macro_dir.mkdir(parents=True, exist_ok=True)
    for symbol, group in ds["ohlcv"].to_pandas().groupby("symbol"):
        group.drop(columns=["symbol", "data_type"]).to_csv(ohlcv_dir / f"{symbol}_daily.csv", index=False)
    for name, group in ds["macro"].to_pandas().groupby("symbol"):
        group.drop(columns=["symbol", "data_type"]).to_csv(macro_dir / f"{name}_daily.csv", index=False)

# World model checkpoint
Path("checkpoints/world_model").mkdir(parents=True, exist_ok=True)
hf_hub_download(
    repo_id="sarthakbiswas/stock-trader-market-data",
    filename="best_transformer.pt",
    repo_type="dataset",
    local_dir="checkpoints/world_model",
)

print(f"Market data: {len(list(ohlcv_dir.glob('*.csv')))} stocks")
print("World model checkpoint downloaded")
print("Setup complete")

## 2. The Environment: What the Agent Sees

The environment generates daily market observations with technical indicators. The agent reads text, not numbers — making it natural for LLMs.

In [ ]:
import sys
sys.path.insert(0, ".")

from training.gym_wrapper import StockTradingGymEnv

# Static replay environment
env = StockTradingGymEnv(task_id="single_stock", seed=42, obs_mode="text", simulator_mode="replay")
obs, info = env.reset()

print("=" * 60)
print("OBSERVATION (what the LLM agent sees each day)")
print("=" * 60)
print(obs[:800])
print("...")
print()
print(f"Portfolio value: Rs{info['portfolio_value']:,.0f}")
print(f"Cash: Rs{info['cash']:,.0f}")
print(f"Task: {info['task_id']}, Day 1/{info['total_days']}")
env.close()

## 3. Neural World Model: The Environment's Brain

The causal transformer (1.22M params) learns market dynamics from 264K rows of real Indian stock data. It generates realistic OHLCV prices — the agent can't tell if data is from CSV or neural network.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from server.neural_simulator import NeuralSimulator

# Generate two episodes with different seeds — both stochastic
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for idx, seed in enumerate([42, 99]):
    sim = NeuralSimulator(task_id="single_stock", seed=seed, temperature=1.0)
    sim.reset()
    symbol = sim.symbols[0]

    gen_df = sim._generated[symbol]
    lookback = 100
    episode_prices = gen_df["close"].iloc[lookback:].values
    seed_prices = gen_df["close"].iloc[lookback-20:lookback].values

    ax = axes[idx]
    ax.plot(range(-20, 0), seed_prices, "b-", alpha=0.5, label="Real (seed)")
    ax.plot(range(0, len(episode_prices)), episode_prices, "r-", linewidth=2, label="Generated")
    ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
    ax.set_title(f"Neural Env Episode (seed={seed})")
    ax.set_xlabel("Day")
    ax.set_ylabel("Price (Rs)")
    ax.legend()

plt.suptitle("Every episode is different — the agent can't memorize", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Model stats
error = sim.compute_prediction_error()
print(f"\nWorld Model: {sum(p.numel() for p in sim._model.parameters()):,} parameters")
print(f"Volatility ratio: {error.get('volatility_ratio', 'N/A')}")
print(f"Direction accuracy: {error.get('direction_accuracy', 'N/A')}")

## 4. Agent Inference: Before vs After Training

Side-by-side comparison of the base model (untrained) vs our best GRPO model. The base model can't follow the output format. The trained model reasons about indicators and makes informed trading decisions.

> **Requires GPU** or HF Inference API. If no GPU available, pre-recorded outputs are shown below.

In [ ]:
# Load the best model (GRPO neural) and run inference
# Falls back to pre-recorded output if GPU not available

USE_LIVE_INFERENCE = torch.cuda.is_available()

SYSTEM_PROMPT = (
    "You are an expert stock trader operating in the Indian equity market.\n"
    "You receive daily market observations with technical indicators and must decide on a trading action.\n\n"
    "Rules:\n"
    "- Respond with EXACTLY one action on the last line\n"
    "- Valid actions: HOLD, BUY, SELL, BUY <SYMBOL>, SELL <SYMBOL>, BUY <SYMBOL> <FRACTION>\n"
    "- Before your action, briefly explain your reasoning in 1-2 sentences inside <think> tags\n\n"
    "Example response:\n"
    "<think>RSI is 25 (oversold) and MACD just turned bullish. Volume is spiking. Good entry point.</think>\n"
    "BUY RELIANCE 0.5"
)

# Get a sample observation for inference
env = StockTradingGymEnv(task_id="single_stock", seed=42, obs_mode="text", simulator_mode="replay")
sample_obs, _ = env.reset()
env.close()

if USE_LIVE_INFERENCE:
    print("GPU detected -- running live inference\n")

    # Install Unsloth for efficient 4-bit loading
    import subprocess
    subprocess.run(["pip", "install", "-q", "unsloth"], capture_output=True)

    from unsloth import FastLanguageModel
    from unsloth.chat_templates import get_chat_template

    # Load GRPO neural model (best: 0.537 on neural env)
    model, tokenizer = FastLanguageModel.from_pretrained(
        "sarthakbiswas/stock-trader-grpo-neural-model",
        max_seq_length=1024,
        load_in_4bit=True,
    )
    tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
    FastLanguageModel.for_inference(model)

    def run_inference(observation):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Here is today's market data:\n\n{observation}\n\nWhat is your trading action?"},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7,
                                     do_sample=True, pad_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    trained_response = run_inference(sample_obs)
else:
    print("No GPU -- showing pre-recorded outputs\n")
    trained_response = (
        "<think>RSI at 28.3 is in oversold territory and MACD histogram is turning positive, "
        "suggesting a bullish reversal. Bollinger position near lower band confirms the stock is "
        "undervalued relative to recent range. Volume spike at 1.4x average shows institutional "
        "interest. Risk is moderate with sideways regime.</think>\n"
        "BUY RELIANCE 0.5"
    )

# Display comparison
base_response = (
    "Looking at the current market data for RELIANCE, I can see several key indicators:\n\n"
    "The RSI is at 28.3, which suggests the stock is in oversold territory. "
    "The MACD shows a bearish crossover but the histogram is narrowing, which could indicate "
    "a potential reversal. The Bollinger Bands show the price is near the lower band...\n\n"
    "(continues for 500+ tokens with no actionable output)"
)

print("=" * 60)
print("BASE MODEL (DeepSeek 7B zero-shot) -- Score: 0.300")
print("=" * 60)
print(base_response)
print()
print("=" * 60)
print("GRPO MODEL (trained against neural env) -- Score: 0.537")
print("=" * 60)
print(trained_response)
print()
print("-" * 60)
print("The base model generates analysis but no action (defaults to HOLD).")
print("The trained model reasons concisely and outputs a clear action.")

## 5. Run a Full Episode: Agent vs Neural Environment

Watch the trained agent play a complete 20-day trading episode against the neural environment. Each day it reads the observation, reasons about indicators, and decides to BUY, SELL, or HOLD.

In [ ]:
# Run a full episode (live if GPU available, pre-recorded otherwise)

if USE_LIVE_INFERENCE:
    env = StockTradingGymEnv(task_id="single_stock", seed=55, obs_mode="text", simulator_mode="neural")
    obs, info = env.reset()
    initial_value = info["portfolio_value"]

    print(f"Starting episode: Rs{initial_value:,.0f} capital, 20 days")
    print("-" * 60)

    for day in range(20):
        response = run_inference(obs)
        # Parse action from response
        action = "HOLD"
        for line in reversed(response.strip().split("\n")):
            if line.strip().upper().startswith(("BUY", "SELL", "HOLD")):
                action = line.strip()
                break

        obs, reward, done, _, info = env.step(action)

        if day < 5 or done:  # Show first 5 days + final
            print(f"Day {day+1:2d} | Action: {action:20s} | Value: Rs{info['portfolio_value']:>10,.0f}")

        if done:
            break

    final_value = info["portfolio_value"]
    episode_return = (final_value - initial_value) / initial_value * 100
    print("-" * 60)
    print(f"Final score: {info['score']:.3f}")
    print(f"Return: {episode_return:+.2f}%")
    print(f"Portfolio: Rs{initial_value:,.0f} -> Rs{final_value:,.0f}")
    env.close()
else:
    # Pre-recorded episode results
    print("Pre-recorded episode (GRPO model vs neural env, seed=55)")
    print("-" * 60)
    print("Day  1 | Action: HOLD                 | Value: Rs   100,000")
    print("Day  2 | Action: BUY RELIANCE 0.5     | Value: Rs   100,150")
    print("Day  3 | Action: HOLD                 | Value: Rs   100,820")
    print("Day  4 | Action: HOLD                 | Value: Rs   101,200")
    print("Day  5 | Action: SELL RELIANCE        | Value: Rs   101,350")
    print("Day 20 | Action: HOLD                 | Value: Rs   102,100")
    print("-" * 60)
    print("Final score: 0.600")
    print("Return: +2.10%")
    print("Portfolio: Rs100,000 -> Rs102,100")

## 6. Reward Design & Safeguards

The environment prevents reward hacking through multiple independent mechanisms.

In [ ]:
from server.mistake_tracker import MistakeTracker, MistakeType

print("ANTI-REWARD-HACKING SAFEGUARDS")
print("=" * 60)

print("""
1. SIGNAL-BASED HOLD
   HOLD is not free. The environment classifies every HOLD:
   - Justified patience (RSI neutral): +0.01 reward
   - Missed opportunity (RSI < 25 or > 75 ignored): -0.15 penalty
   - Holding a loser (position at -5%): -0.10 penalty

2. MISTAKE TRACKER (7 error types, real-time detection)""")

for mt in MistakeType:
    print(f"   - {mt.value}")

print("""
3. MULTIPLE INDEPENDENT REWARD FUNCTIONS
   - format_gate: penalty-only gate (0 valid, -1 invalid)
   - trading_reward: 30% step-level P&L + 70% episode return
   - Anti-HOLD collapse: rolling window penalty above 75% HOLD

4. GRADING (verifiable, objective)
   - single_stock: agent return vs buy-and-hold benchmark
   - portfolio: 60% Sharpe + 25% discipline + 15% activity
   - full_autonomous: 35% return + 25% risk-adjusted + 25% regime + 15% risk mgmt

5. ENVIRONMENT CONSTRAINTS
   - Transaction costs + slippage (0.1-0.2%)
   - Position limits (max 20-40% per stock)
   - Trade limits (5-10 per day)
   - Regime gate (blocks trading when market breadth is weak)
""")

print("These safeguards ensure the agent learns to TRADE, not to hack the reward.")

## 7. Learning Curve: Evidence of Improvement

The complete training progression across 8+ experiments, showing clear improvement from base model to GRPO.

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# Load real training logs
with open("results/sft_v3_training_log.json") as f:
    sft_log = json.load(f)
with open("results/grpo_neural_training_log.json") as f:
    grpo_log = json.load(f)

C_SFT, C_GRPO, C_FAIL, C_GRAY = "#3498db", "#2ecc71", "#e74c3c", "#95a5a6"

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.patch.set_facecolor("white")
for ax in axes.flat:
    ax.set_facecolor("#fafafa")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# --- Panel A: SFT Loss ---
steps = [e["step"] for e in sft_log]
losses = [e["loss"] for e in sft_log]
axes[0,0].plot(steps, losses, color=C_SFT, linewidth=2.5)
axes[0,0].fill_between(steps, losses, alpha=0.08, color=C_SFT)
axes[0,0].axvline(x=200, color=C_GRPO, linestyle="--", alpha=0.7)
best_idx = next(i for i, e in enumerate(sft_log) if e["step"] == 200)
axes[0,0].plot(200, losses[best_idx], "o", color=C_GRPO, markersize=10, zorder=5)
axes[0,0].annotate("Best checkpoint\nscore 0.399", xy=(200, losses[best_idx]),
    xytext=(120, losses[best_idx]+0.15), fontsize=8.5, fontweight="bold", color=C_GRPO,
    arrowprops=dict(arrowstyle="->", color=C_GRPO, lw=1.2))
axes[0,0].annotate("Step 352: loss 2.01\nbut score dropped to 0.383",
    xy=(200, losses[-1]), xytext=(140, 2.65), fontsize=7.5, color=C_FAIL, style="italic",
    arrowprops=dict(arrowstyle="->", color=C_FAIL, lw=1, alpha=0.6))
axes[0,0].set_xlabel("Training Step"); axes[0,0].set_ylabel("Loss")
axes[0,0].set_title("SFT: Teaching the Model to Trade", fontweight="bold")

# --- Panel B: KL Divergence ---
g_steps = [e["step"] for e in grpo_log]
kl = [e["kl"] for e in grpo_log]
axes[0,1].plot(g_steps, kl, color=C_GRPO, linewidth=2)
axes[0,1].fill_between(g_steps, kl, alpha=0.1, color=C_GRPO)
axes[0,1].axhline(y=3.0, color=C_FAIL, linestyle="--", linewidth=1.5, alpha=0.7)
axes[0,1].text(155, 3.15, "GRPO v3 collapsed here (KL=4.2)", fontsize=8, color=C_FAIL, fontweight="bold", ha="center")
axes[0,1].fill_between([0, 300], [0, 0], [0.5, 0.5], alpha=0.05, color=C_GRPO)
axes[0,1].text(150, 0.05, "Safe zone (beta=0.05)", fontsize=8, color=C_GRPO, alpha=0.7, ha="center")
axes[0,1].set_xlabel("Training Step"); axes[0,1].set_ylabel("KL Divergence")
axes[0,1].set_title("GRPO: KL Divergence Under Control", fontweight="bold")
axes[0,1].set_ylim(-0.05, 3.8)

# --- Panel C: Trading Reward ---
r_mean = [e["rewards/trading_reward/mean"] for e in grpo_log]
r_std = [e["rewards/trading_reward/std"] for e in grpo_log]
axes[1,0].fill_between(g_steps, [m-s for m,s in zip(r_mean,r_std)],
    [m+s for m,s in zip(r_mean,r_std)], alpha=0.15, color=C_GRPO, label="Reward std")
axes[1,0].plot(g_steps, r_mean, color=C_GRPO, linewidth=2, label="Trading reward")
smoothed = np.convolve(r_mean, np.ones(5)/5, mode="valid")
axes[1,0].plot(g_steps[4:], smoothed, color="#27ae60", linewidth=2.5, alpha=0.8, label="Trend (5-step avg)")
axes[1,0].axhline(y=0, color=C_GRAY, linewidth=0.8, alpha=0.5)
axes[1,0].set_xlabel("Training Step"); axes[1,0].set_ylabel("Trading Reward")
axes[1,0].set_title("GRPO: Trading Reward Signal", fontweight="bold")
axes[1,0].legend(fontsize=8, loc="lower left")

# --- Panel D: Learning Curve ---
models = ["Base\nDeepSeek 7B", "SFT v3\n(distilled)", "GRPO Neural\n(BEST)"]
static = [0.300, 0.399, 0.470]; neural = [0.298, 0.417, 0.537]
x = np.arange(len(models)); w = 0.32
b1 = axes[1,1].bar(x-w/2, static, w, label="Static Env", color=C_SFT, edgecolor="white", linewidth=1.5)
b2 = axes[1,1].bar(x+w/2, neural, w, label="Neural Env", color=C_GRPO, edgecolor="white", linewidth=1.5)
axes[1,1].axhline(y=0.300, color=C_FAIL, linestyle="--", linewidth=1.2, alpha=0.5)
for bar in b1:
    axes[1,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.008,
        f"{bar.get_height():.3f}", ha="center", fontsize=9, fontweight="bold", color=C_SFT)
for bar in b2:
    axes[1,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.008,
        f"{bar.get_height():.3f}", ha="center", fontsize=9, fontweight="bold", color="#1a8a4a")
axes[1,1].annotate("", xy=(2.16,0.53), xytext=(0.16,0.30),
    arrowprops=dict(arrowstyle="->", color=C_GRAY, lw=2, connectionstyle="arc3,rad=0.15"))
axes[1,1].text(1.1, 0.48, "+79%", fontsize=11, fontweight="bold", color=C_GRAY, ha="center")
axes[1,1].set_xticks(x); axes[1,1].set_xticklabels(models, fontsize=10)
axes[1,1].set_ylabel("Score"); axes[1,1].set_ylim(0.2, 0.62)
axes[1,1].set_title("Result: Base 0.300 → GRPO 0.537", fontweight="bold")
axes[1,1].legend(fontsize=9, loc="upper left")

fig.suptitle("Training the Trading Agent: SFT → GRPO Against Neural Environment",
    fontsize=14, fontweight="bold", y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("results/training_curves_final.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print(f"\nImprovement: Base 0.300 → SFT 0.417 → GRPO 0.537")
print(f"Total improvement: +79% over base model (neural env)")
print(f"RL improvement: +29% over SFT (neural env)")
print(f"KL stayed under 0.35 (previous GRPO v3 reached 4.2 and collapsed)")

## 8. The Failure Gallery: What Didn't Work (and Why)

4 GRPO failures, each teaching a fundamental lesson about RL reward design.

In [ ]:
failures = [
    {
        "name": "GRPO v1: The HOLD Collapse",
        "score": 0.300,
        "problem": "85% of actions were HOLD. The model learned that doing nothing = zero risk.",
        "root_cause": "HOLD had zero cost. BUY/SELL could go negative. Rational agent chooses inaction.",
        "fix": "Signal-based HOLD: ignoring RSI < 25 or > 75 now costs -0.15. Justified patience gets +0.01.",
    },
    {
        "name": "GRPO v2: The Format Hack",
        "score": 0.326,
        "problem": "84% of reward came from formatting. 0% from trading.",
        "root_cause": "Format compliance (+1.0 reward) dominated trading signal. Model optimized for looking smart.",
        "fix": "Format is now a gate (0/-1), not a reward source. All positive reward from trading only.",
    },
    {
        "name": "GRPO v3: The KL Catastrophe",
        "score": 0.301,
        "problem": "Best training distribution (59% HOLD, 25% BUY, 16% SELL) but worst eval score.",
        "root_cause": "KL divergence climbed to 4.2. Model forgot its SFT knowledge while chasing RL reward.",
        "fix": "KL early stopping at 3.0. Higher beta (0.05). Shorter training (300 steps, not 1000).",
    },
    {
        "name": "SFT v3: The Alignment Tax",
        "score": 0.383,
        "problem": "Step 352 (lower loss) scored WORSE than step 200 (higher loss).",
        "root_cause": "Continued training = memorization, not generalization. Lower loss != better trading.",
        "fix": "Select checkpoints by task score, not by training loss. Peak was at 57% through training.",
    },
]

for i, f in enumerate(failures, 1):
    print(f"{'=' * 60}")
    print(f"FAILURE {i}: {f['name']} (score: {f['score']})")
    print(f"{'=' * 60}")
    print(f"  Problem:    {f['problem']}")
    print(f"  Root cause: {f['root_cause']}")
    print(f"  Fix:        {f['fix']}")
    print()

print("Each failure directly informed the GRPO neural config that scored 0.537.")

## 9. Architecture & Pipeline Summary

In [ ]:
print("""
ARCHITECTURE
============

Real Market Data (264K rows, 109 NIFTY stocks, 2015-2026)
        |
        v
Causal Transformer World Model (1.22M params, 4 layers, MDN output)
  - Volatility: 0.94x reality
  - Trained in 7 min on A5000
  - Drop-in replacement for CSV replay
        |
        v
Generated OHLCV --> Feature Engine --> Text Observation
  (RSI, MACD, Bollinger, trend, momentum, regime, volume)
        |
        v
LLM Agent (DeepSeek-R1 7B, QLoRA r=16)
  - Reads text observations
  - Outputs: <think>reasoning</think> ACTION
        |
        v
Environment Grader (verifiable score 0.0-1.0)


TRAINING PIPELINE (using TRL + Unsloth)
=========================================

1. SFT (Supervised Fine-Tuning)
   - 10K examples via reverse distillation (GPT-4o-mini)
   - Conservative: lr=5e-6, r=16, q/k/v/o only
   - Result: 0.300 -> 0.417

2. RAFT (Rejection Sampling Fine-Tuning)
   - 100 episodes against neural env
   - Keep winners (score > 0.5): 32/100 episodes
   - SFT on 640 winning examples

3. GRPO (Group Relative Policy Optimization)
   - 1000 prompts from neural environment
   - TRL GRPOTrainer + Unsloth
   - 300 steps, G=4, beta=0.05
   - Result: 0.417 -> 0.537


SAFEGUARDS
==========
- Signal-based HOLD (no free inaction)
- 7-type mistake tracker (wired into rewards)
- Multiple independent reward functions
- Adaptive curriculum (5 difficulty tiers)
- KL early stopping (prevents knowledge loss)


REFERENCES
==========
- Ha & Schmidhuber (2018): World Models — V-M-C architecture
- Trading-R1 (arXiv 2509.11420): Reverse distillation + RL curriculum
- LIMA (Zhou et al. 2023): Quality > quantity in training data
- Pikus et al.: Hard-example GRPO filtering
- Dr. GRPO (arXiv 2503.20783): Batch reward scaling
- PAIRED (Dennis et al. 2020): Regret-driven environment design
""")

print(f"Repository: github.com/sarthakbiswas97/stock-trader-env")
print(f"Models: huggingface.co/sarthakbiswas")
print(f"Tests: 247 | Coverage: 80%+ | CI: GitHub Actions")
print(f"\nThe environment has a brain. The agent has to earn its score.")